In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
pd.options.display.float_format = '{:.2f}'.format
df = pd.read_json('../data/auto.json')

создаем сэпмл, получаем только уникальные комбинации, реплейс = тру дает выбрать комбинации нескольок раз, делаем это для возвратов и штрафов, меняем индексы, создаем копию и сбрасываем индексы и добавляем к комбо случайный штраф и возврат

In [4]:
np.random.seed(21)
cars = df[['CarNumber', 'Make', 'Model']].drop_duplicates()
sample_cars = cars.sample(n=200, replace=True, random_state=21)

sample_refund = df['Refund'].sample(n=200, replace=True, random_state=21).reset_index(drop=True)
sample_fine = df['Fines'].sample(n=200, replace=True, random_state=21).reset_index(drop=True)

sample_df = sample_cars.reset_index(drop=True).copy()
sample_df['Refund'] = sample_refund
sample_df['Fines'] = sample_fine

объединяем данные и добавляем столбец с годом к данным

In [6]:
df_combined = pd.concat([df, sample_df], ignore_index=True)

np.random.seed(21)
years = np.random.randint(1980, 2020, size=len(df_combined))
df_fines = df_combined.copy()
df_fines['Year'] = years

создаем датафрейм с 5 новыми записями о штрафах и добавляем

In [8]:
new_fines = pd.DataFrame({
    'CarNumber': ['XYZ999', 'ABC123', 'DEF456', 'GHI789', 'JKL012'],
    'Make': ['Toyota', 'Honda', 'Ford', 'BMW', 'Mercedes'],
    'Model': ['Camry', 'Civic', 'Focus', 'X5', 'C-Class'],
    'Refund': [0, 1, 0, 1, 0],
    'Fines': [250.00, 150.50, 300.00, 500.75, 200.25],
    'Year': [1988, 1999, 2000, 2001, 2002],})
df_fines = pd.concat([df_fines, new_fines], ignore_index=True)

загружаем фамилии из json, очищаем фамилии от лишних символов

In [10]:
surnames_df = pd.read_json('../data/surname.json')
surnames_clean = [str(row[0]).replace(',', '').replace('(', '').replace(')', '') for row in surnames_df.values[1:]]

получаем уникальные номера автомобилей, фамилии владельцев назначаем рандомно и создаем датафрейм

In [12]:
car_numbers = df['CarNumber'].unique()
np.random.seed(21)
owner_names = np.random.choice(surnames_clean, size=len(car_numbers), replace=True)

df_owners = pd.DataFrame({
    'CarNumber': car_numbers,
    'SURNAME': owner_names})

In [13]:
len(df_owners)

531

удаляем последние 20 записей, добавляем 3 новых владельцев и объединяем данные

In [15]:
df_owners = df_owners.iloc[:-20]
new_owners = pd.DataFrame({
    'CarNumber': ['MNO345', 'PQR678', 'STU901'],
    'SURNAME': ['Taylor', 'Thomas', 'Moore']})
df_owners = pd.concat([df_owners, new_owners], ignore_index=True)

иннер - только общие записи, аутер - все записи из двух таблиц, лефт - все записи из штрафов+совпадения из владельцев, райт - все записи из владельцев+ совпадения из штрафов 

In [17]:
df_inner = pd.merge(df_fines, df_owners, on='CarNumber', how='inner')
df_outer = pd.merge(df_fines, df_owners, on='CarNumber', how='outer')
df_left = pd.merge(df_fines, df_owners, on='CarNumber', how='left')
df_right = pd.merge(df_fines, df_owners, on='CarNumber', how='right')

создаем pivot с суммой штрафов по моделям и годам, отсутсвующие значения заполняем нанами и отображаем годы с 1980 по 2019

In [19]:
pd.pivot_table(
   df_fines,
    values='Fines',
    index=['Make', 'Model'],
    columns='Year',
    aggfunc='sum',
    fill_value=np.nan).reindex(columns=range(1980, 2020), fill_value=np.nan)

Year                   1980      1981      1982     1983      1984      1985  \
Make       Model                                                               
BMW        X5           NaN       NaN       NaN      NaN       NaN       NaN   
Ford       Focus   56489.17 398589.17 140383.76 62300.00 112494.59 189583.76   
           Mondeo       NaN       NaN       NaN      NaN       NaN       NaN   
Honda      Civic        NaN       NaN       NaN      NaN       NaN       NaN   
Mercedes   C-Class      NaN       NaN       NaN      NaN       NaN       NaN   
Skoda      Octavia  1900.00       NaN   6900.00 11594.59       NaN  10294.59   
Toyota     Camry   28500.00   8594.59       NaN  7200.00       NaN       NaN   
           Corolla      NaN       NaN   2000.00   800.00       NaN       NaN   
Volkswagen Golf    30900.00       NaN       NaN  8594.59    300.00  24000.00   
           Jetta        NaN       NaN       NaN      NaN       NaN       NaN   
           Passat   6500.00   1600.00       NaN  3200.00  10000.00   5000.00   
           Touareg      NaN       NaN       NaN      NaN       NaN   5800.00   

Year                    1986      1987     1988      1989  ...      2010  \
Make       Model                                           ...             
BMW        X5            NaN       NaN      NaN       NaN  ...       NaN   
Ford       Focus   104994.59 132800.00 95489.17 125700.00  ... 120183.76   
           Mondeo        NaN       NaN      NaN   8600.00  ...       NaN   
Honda      Civic         NaN       NaN      NaN       NaN  ...       NaN   
Mercedes   C-Class       NaN       NaN      NaN       NaN  ...       NaN   
Skoda      Octavia    600.00   5200.00   500.00  91400.00  ...   3100.00   
Toyota     Camry         NaN       NaN   250.00  22400.00  ...       NaN   
           Corolla       NaN   8000.00      NaN   4000.00  ...  24000.00   
Volkswagen Golf          NaN   9300.00      NaN   5800.00  ...       NaN   
           Jetta         NaN       NaN      NaN       NaN  ...       NaN   
           Passat   15000.00  12300.00      NaN       NaN  ...   2800.00   
           Touareg       NaN       NaN      NaN       NaN  ...   6300.00   

Year                   2011      2012      2013      2014      2015     2016  \
Make       Model                                                               
BMW        X5           NaN       NaN       NaN       NaN       NaN      NaN   
Ford       Focus   86689.17 120200.00 149294.59 157494.59 210789.17 83694.59   
           Mondeo       NaN  34400.00       NaN       NaN       NaN 48100.00   
Honda      Civic        NaN       NaN       NaN       NaN       NaN      NaN   
Mercedes   C-Class      NaN       NaN       NaN       NaN       NaN      NaN   
Skoda      Octavia   500.00    500.00  19594.59   3300.00  46394.59   300.00   
Toyota     Camry    3300.00  10594.59       NaN       NaN       NaN      NaN   
           Corolla  8594.59       NaN       NaN       NaN       NaN      NaN   
Volkswagen Golf      300.00       NaN       NaN       NaN   2300.00      NaN   
           Jetta        NaN       NaN       NaN       NaN       NaN      NaN   
           Passat       NaN       NaN       NaN       NaN    600.00  2100.00   
           Touareg      NaN       NaN       NaN   1300.00    500.00      NaN   

Year                    2017      2018      2019  
Make       Model                                  
BMW        X5            NaN       NaN       NaN  
Ford       Focus   268200.00 283594.59 117100.00  
           Mondeo        NaN       NaN       NaN  
Honda      Civic         NaN       NaN       NaN  
Mercedes   C-Class       NaN       NaN       NaN  
Skoda      Octavia   4000.00 156200.00   9500.00  
Toyota     Camry     1000.00  13000.00  18100.00  
           Corolla   9600.00       NaN       NaN  
Volkswagen Golf          NaN       NaN       NaN  
           Jetta         NaN       NaN       NaN  
           Passat        NaN       NaN       NaN  
           Touareg       NaN  

In [20]:
df_fines.to_csv('fines.csv', index=False)
df_owners.to_csv('owners.csv', index=False)
#df_fines.head()

In [21]:
idx_to_fill = df_combined[df_combined['Model'].isna()].index[0]
df_combined.at[idx_to_fill, 'Model'] = 'Model'

In [22]:
df_combined.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        914
dtype: int64

In [23]:
df_fines.count()

CarNumber    930
Refund       930
Fines        930
Make         930
Model        918
Year         930
dtype: int64

In [24]:
df_owners['SURNAME'].head()

0    RICHARDSON
1          ROSS
2        MORGAN
3        BAILEY
4         LOPEZ
Name: SURNAME, dtype: object

In [25]:
len(df_fines) 

930